# Análise de Duplicação Intragenômica e Auto-Sintenia (Self-Synteny) com JCVI (MCscan)

Este notebook descreve o pipeline para identificar possíveis duplicações gênicas e regiões de auto-sintenia (parálogos/WGD/duplicações segmentais) nos genomas de *V. peripherica* (*Vper*) e *V. tubiflora* (*Vtub*) utilizando a ferramenta **JCVI (MCscan em Python)**.

**Nova Organização Isolada:**
Para evitar conflitos de nomes padrão (collisions) entre inputs e outputs gerados pelo JCVI, rodamos as análises intragenômicas em ambientes isolados dedicados:
- `intragenomic/Vper/` para *V. peripherica*
- `intragenomic/Vtub/` para *V. tubiflora*

**Pré-requisitos:**
- O ambiente conda `jcviEnv` deve estar ativo.
- Os links simbólicos locais para os arquivos BED e CDS devem estar presentes nos diretórios isolados correspondentes (já criados e automatizados pelo script `prepare_files.py`).

## 1. Auto-Sintenia de *V. peripherica* (Vper vs Vper)

Acessamos a pasta isolada de `Vper` para rodar o pipeline, garantindo que os arquivos intermediários sejam gerados localmente e não causem colisões com a análise de `Vtub`.

In [ ]:
import os
# Garante que o notebook mude para o diretório de execução correto
notebook_dir = os.getcwd()
target_dir = os.path.abspath(os.path.join(notebook_dir, "..", "intragenomic", "Vper"))
os.chdir(target_dir)
print(f"Diretório de execução atual: {os.getcwd()}")

Rodamos o alinhamento local de `Vper` contra ele mesmo. Para manter a organização de arquivos pesados, direcionamos a indexação do LAST para o diretório centralizado `last_db/` usando `--path=../../last_db`.

In [ ]:
# Rodar o cálculo de auto-sintenia usando o last_db centralizado
!python -m jcvi.compara.synteny screen Vper.bed Vper.bed --last --path=../../last_db

Para identificar duplicações intragenômicas mais específicas ou duplicações em tandem/parálogos contendo assinaturas de interesse (como blocos que possuem genes `P98L0`), podemos rodar a catalogação de parálogos/ortólogos:

In [ ]:
# Gerar catálogo de ortologia/paralogia interno
!python -m jcvi.compara.catalog ortholog Vper Vper --no_strip_names --path=../../last_db

### Visualização de Auto-Sintenia e Duplicações de Vper

Podemos visualizar o Dotplot gerado:

In [ ]:
# Visualizar o Dotplot de auto-sintenia de Vper
!python -m jcvi.graphics.dotplot Vper.Vper.anchors

---

## 2. Auto-Sintenia de *V. tubiflora* (Vtub vs Vtub)

Navegamos agora para o diretório isolado de `Vtub`.

In [ ]:
import os
# Mudar para o diretório isolado de Vtub
notebook_dir = os.path.abspath(os.path.join(os.getcwd(), "..", "Vtub"))
os.chdir(notebook_dir)
print(f"Diretório de execução atual: {os.getcwd()}")

In [ ]:
# Rodar o cálculo de auto-sintenia para Vtub direcionando indexação do LAST para last_db
!python -m jcvi.compara.synteny screen Vtub.bed Vtub.bed --last --path=../../last_db

Para `Vtub`, identificamos uma duplicação segmental muito rica entre os cromossomos `vt_chr1A` e `vt_chr2A` contendo **15 pares de genes parálogos** no alinhamento de auto-sintenia.

Para renderizar essa macrosintenia local (linhas finas conectando os parálogos):

In [ ]:
# Renderizar o plot usando as configurações de layout e blocos criadas na pasta layouts/
# Nota: Sempre use a flag '--notex' para evitar erros de renderização LaTeX locais
!python -m jcvi.graphics.synteny layouts/intragenome.blocks layouts/intragenome.layout --notex

A figura final gerada estará salva de forma limpa e organizada em `results/intragenome.pdf`.

### Visualização do Dotplot de Vtub

In [ ]:
# Visualizar o Dotplot de auto-sintenia de Vtub
!python -m jcvi.graphics.dotplot Vtub.Vtub.anchors